In [31]:
# Uncomment and run this cell if you're on Colab or Kaggle
# !git clone https://github.com/nlp-with-transformers/notebooks.git
# %cd notebooks
# from install import *
# install_requirements()

In [32]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [33]:
# hide
from utils import *
setup_chapter()

No GPU was detected! This notebook can be *very* slow without a GPU 🐢
Using transformers v4.57.6
Using datasets v4.5.0


# Making Transformers Efficient in Production

<img alt="Scaling BERT at Roblox" caption="How Roblox scaled BERT with knowledge distillation, dynamic padding, and weight quantization (photo courtesy of Roblox employees Quoc N. Le and Kip Kaehler)" src="images/chapter08_roblox.png" id="roblox"/>

## Intent Detection as a Case Study

<img alt="Out of Scope Query" width="400" caption="Three exchanges between a human (right) and a text-based assistant (left) for personal finance (courtesy of Stefan Larson et al.)" src="images/chapter08_oos.png" id="oos"/> 

In [34]:
#hide_output
from transformers import pipeline

bert_ckpt = "transformersbook/bert-base-uncased-finetuned-clinc"
pipe = pipeline("text-classification", model=bert_ckpt)

```text
BERT preentrenado
        │
        ▼
Fine-tuning sobre CLINC150
        │
        ▼
Modelo capaz de clasificar intenciones
```

| Etapa | Qué aprende el modelo |
|-------|------------------------|
| **Pretraining** | Comprende el lenguaje: gramática, vocabulario y relaciones entre palabras. |
| **Fine-tuning en CLINC150** | Aprende la tarea de clasificación de intenciones (*Intent Classification*). |
| **Resultado** | Modelo capaz de predecir la intención de una consulta o frase. |

In [35]:
query = """Hey, I'd like to rent a vehicle from Nov 1st to Nov 15th in 
Paris and I need a 15 passenger van"""
pipe(query)

[{'label': 'car_rental', 'score': 0.5490034222602844}]

## Creating a Performance Benchmark

Esta clase crea un marco común para medir todas las versiones del modelo. La idea es que más adelante puedas pasarle el BERT original, el modelo destilado o el cuantizado y evaluarlos exactamente con las mismas pruebas.

In [36]:
class PerformanceBenchmark:
    # El constructor
    def __init__(self, pipeline, dataset, optim_type="BERT baseline"):
        self.pipeline = pipeline
        # pipeline: el modelo preparado para hacer inferencias.
        self.dataset = dataset
        # dataset: los ejemplos con los que se medirá.
        self.optim_type = optim_type
        # optim_type: el nombre de la versión evaluada.
    
    # Los 3 métodos de medición:
    
    # calidad de las predicciones
    def compute_accuracy(self):
        # We'll define this later
        pass # este método existe pero no hace nada todavía   

    # tamaño del modelo
    def compute_size(self):
        # We'll define this later
        pass

    # tiempo de inferencia del pipeline
    def time_pipeline(self):
        # We'll define this later
        pass
    
    def run_benchmark(self):
        metrics = {}
        # ejecuta compute_size y guarda su resultado bajo el nombre del modelo
        metrics[self.optim_type] = self.compute_size()
        # actualiza el diccionario metrics incoporando clave valor de time_pipeline
        metrics[self.optim_type].update(self.time_pipeline())
        # actualiza el diccionario añadiendo la métrica de calidad
        metrics[self.optim_type].update(self.compute_accuracy())
        return metrics

In [37]:
#hide_output
# Cargamos el dataset de CLINC150

from datasets import load_dataset

clinc = load_dataset("clinc_oos", "plus")
# este dataset tiene varias configuraciones. Aquí se carga la versión plus que incluye:
    # train
    # validation
    # test
    # ejemplos out-of-scope(OOS)

In [38]:
clinc

DatasetDict({
    train: Dataset({
        features: ['text', 'intent'],
        num_rows: 15250
    })
    validation: Dataset({
        features: ['text', 'intent'],
        num_rows: 3100
    })
    test: Dataset({
        features: ['text', 'intent'],
        num_rows: 5500
    })
})

In [39]:
# dentro del subconjunto test quiero ver el nombre de las columnas
clinc["test"].column_names

['text', 'intent']

In [40]:
# selecciona un ejemplo

sample = clinc["test"][42]
sample

{'text': 'transfer $100 from my checking to saving account', 'intent': 133}

La intención todavía aparece como número entero

In [41]:
# Para pasarlo a texto

intents = clinc["test"].features["intent"] # obtenemos definición de la columna "intent"
intents.int2str(sample["intent"])
# sample["intent"] -> devuelve un integer: la columna "intent" de sample
# intents.int2str(sample["intent"]) --> consulta la tabla de etiquetas del dataset y devuelve el texto de la intención

'transfer'

Para entrenar modelos es mucho más eficiente trabajar con enteros.

In [42]:
# Importamos la librería Evaluate, desarrollada por Hugging Face 
# para calcular métricas de evaluación de modelos.

import evaluate

accuracy_score = evaluate.load("accuracy") # cargamos el objeto para calcular la accuracy



In [43]:
# Implementamos el método compute_accuracy() que antes estaba vacío

def compute_accuracy(self):
    """This overrides the PerformanceBenchmark.compute_accuracy() method"""
    preds, labels = [], [] # creamos dos listas vacías
    # preds: las predicciones del modelo.-> ID's
    # labels: las etiquetas reales del dataset.-> ID's
    for example in self.dataset: # Recorre uno por uno todos los ejemplos del dataset de test.
        pred = self.pipeline(example["text"])[0]["label"]
        # example["text"] -> Obtiene la frase p.ej "How do I change my PIN?"
        # self.pipeline() lo pasa por el modelo BERT. Este modelo devuelve algo así:
            # [
                # {
                #    "label": "change_pin",
                #    "score": 0.998
                #}
            #]
        # la pipeline devuelve una lista aunque sea un resultado -> [0] : 1er resultado
        # ["label"] -> extraemos únicamente "label" porque  es la predicción -> string!
        
        label = example["intent"] # de cada ejemplo del dataset cogemos la respuesta verdadera
        preds.append(intents.str2int(pred))
        # intents.str2int(pred)-> transformamos los preds de string a integer
        # lo añadimos a la lista preds
        labels.append(label) # lo añadimos a la lista labels. Ya es un integer porque intent es integer
    
    # Ahora calculamos la accuracy comparando todas las preds (predicciones) con las labels (etiquetas reales)
    accuracy = accuracy_score.compute(predictions=preds, references=labels)
    # .compute() -> La función recorre ambas listas simultáneamente y compara cada elemento.
        #references-> respuestas correctas
        #predictions-> predicciones
    # devuelve un número -> podemos hacer accuracy[accuracy]
    print(f"Accuracy on test set - {accuracy['accuracy']:.3f}")
    return accuracy

PerformanceBenchmark.compute_accuracy = compute_accuracy
# reemplazar el método compute_accuracy de la clase PerformanceBenchmark por la función que acabas de definir
# "A partir de ahora, cuando alguien llame a compute_accuracy() sobre un objeto PerformanceBenchmark, 


In [44]:
# para inspeccionar un parámetro concreto del modelo

list(pipe.model.state_dict().items())[42]

# pipe.model-> Accedes al modelo de PyTorch que hay dentro de la pipeline.
# state_dict() -> Devuelve un diccionario con todos los pesos aprendidos del modelo.
# .items() -> Convierte el diccionario en pares (nombre, tensor)
# list() -> Lo convierte en lista
#[42] -> Selecciona el elemento 42 de la lista


('bert.encoder.layer.2.attention.self.value.bias',
 tensor([-2.7834e-02,  4.9434e-02,  8.3551e-02,  4.1092e-02,  6.0157e-01,
          1.1774e-01, -5.2112e-02, -6.5143e-02, -2.9358e-02, -4.2250e-02,
          7.9177e-02,  8.0409e-02,  2.9921e-03,  1.7816e-01, -5.0480e-02,
         -1.5634e-01, -2.1707e-02,  1.4381e-02,  2.5132e-02, -2.4110e-02,
         -1.9183e-01, -7.8657e-02,  5.0709e-02,  3.3632e-02, -3.1946e-02,
          1.1616e-01,  9.2720e-02, -1.1787e-01,  2.3233e-01, -1.2678e-02,
         -1.3138e-01, -4.0024e-02,  7.4823e-02, -5.4148e-02, -1.5184e-01,
         -7.4407e-02,  1.1559e-01,  8.2729e-02, -1.3787e-01,  8.3528e-02,
          1.2154e-01,  1.6880e-02, -5.6629e-02, -3.9295e-02,  5.3725e-02,
          6.8602e-02, -1.1294e-01,  4.4001e-02, -2.5884e-01,  1.6767e-01,
          1.8316e-01,  5.6272e-02, -3.6874e-02, -2.7938e-02, -9.3204e-02,
         -7.5239e-03,  4.1141e-02, -1.1542e-02, -9.9749e-02, -3.0910e-02,
          4.1398e-02, -4.4389e-02, -2.6279e-02,  7.2100e-02, 

### ¿Qué se pretende mostrar?

El objetivo es que el lector descubra que un modelo de **PyTorch** está formado por un conjunto de **tensores con nombre**.

Es decir, que pase de pensar:

```text
BERT
```

a pensar:

```text
BERT
│
├── embeddings.word_embeddings.weight
├── embeddings.position_embeddings.weight
├── encoder.layer.0.attention.self.query.weight
├── encoder.layer.0.attention.self.key.weight
├── ...
└── classifier.weight
```

Todos esos tensores (pesos y sesgos del modelo) viven dentro del método:

```python
model.state_dict()
```

Conceptualmente:

```text
BERT
 │
 ▼
state_dict()
 │
 ├── embeddings.word_embeddings.weight
 ├── embeddings.position_embeddings.weight
 ├── embeddings.LayerNorm.weight
 ├── embeddings.LayerNorm.bias
 │
 ├── encoder.layer.0.attention.self.query.weight
 ├── encoder.layer.0.attention.self.query.bias
 ├── encoder.layer.0.attention.self.key.weight
 ├── encoder.layer.0.attention.self.key.bias
 ├── ...
 │
 └── classifier.weight
```

Cada entrada del `state_dict()` es un par:

```text
nombre del tensor
        │
        ▼
tensor de PyTorch (pesos o sesgos)
```

Por tanto, `state_dict()` puede entenderse como **un diccionario que contiene todos los parámetros aprendidos por el modelo**, identificados mediante un nombre único.

### ¿Qué representa cada elemento del `state_dict()`?

Cada elemento del `state_dict()` corresponde a **un único tensor**, no necesariamente a una capa completa.

Por ejemplo, imagina un `state_dict()` simplificado:

```text
[0]  embeddings.word_embeddings.weight
[1]  embeddings.position_embeddings.weight
[2]  embeddings.LayerNorm.weight
[3]  embeddings.LayerNorm.bias
[4]  encoder.layer.0.attention.self.query.weight
[5]  encoder.layer.0.attention.self.query.bias
[6]  encoder.layer.0.attention.self.key.weight
[7]  encoder.layer.0.attention.self.key.bias
...
[42] encoder.layer.2.output.dense.weight
...
```

Aquí, el elemento **42** sería un tensor concreto: los **pesos de una capa lineal** dentro de la tercera capa del encoder (`layer.2`).

---

### Una capa Transformer está formada por muchos tensores

Observa que una sola capa del Transformer está formada por numerosos tensores:

```text
Encoder Layer 2 (tercera capa!)
│
├── attention.self.query.weight
├── attention.self.query.bias
├── attention.self.key.weight
├── attention.self.key.bias
├── attention.self.value.weight
├── attention.self.value.bias
├── attention.output.dense.weight
├── attention.output.dense.bias
├── intermediate.dense.weight
├── intermediate.dense.bias
├── output.dense.weight
├── output.dense.bias
├── LayerNorm.weight
└── LayerNorm.bias
```

Cada una de esas líneas aparece como **un elemento distinto** en el `state_dict()`.

---

### En resumen

- ✅ `encoder.layer.2` → es una **capa Transformer**.
- ✅ `encoder.layer.2.output.dense.weight` → es **un tensor** de esa capa.
- ❌ `[42]` **no identifica una capa**; simplemente es la **posición** de un tensor dentro del diccionario.

En otras palabras, **`state_dict()` no almacena capas, sino los tensores (pesos y sesgos) que componen cada capa**.

Cada elemento del `state_dict()` es una **tupla** formada por una **clave** y un **valor**:

```python
(
    'bert.encoder.layer.2.attention.self.value.bias',
    tensor([...])
)
```

Conceptualmente:

```text
(clave, valor)
      │
      ├── Clave
      │      └── 'bert.encoder.layer.2.attention.self.value.bias'
      │
      └── Valor
             └── tensor([...])
```

- **Clave (`key`)**: nombre único que identifica el tensor dentro del modelo.
- **Valor (`value`)**: el tensor de PyTorch que contiene los pesos o sesgos aprendidos.

In [45]:
import torch # permite guardar el modelo
from pathlib import Path # permite trabajar con archivos

def compute_size(self): # Este método se añadirá a la clase PerformanceBenchmark.
    """This overrides the PerformanceBenchmark.compute_size() method"""
    state_dict = self.pipeline.model.state_dict() # recupera todos los parámetros entrenados del modeo (diccionario de pesos)
    tmp_path = Path("model.pt") # crea el nombre del archivo
    torch.save(state_dict, tmp_path) # se crea el archivo mode.pt que contiene todos los pesos
    # Calculate size in megabytes
    size_mb = Path(tmp_path).stat().st_size / (1024 * 1024)
    # / (1024 * 1024) -convierte bytes en mega byte (1 MB = 1024 × 1024 bytes)
    # Delete temporary file
    tmp_path.unlink() # elimina el archivo porque sólo se necesitaba para medir el tamaño
    print(f"Model size (MB) - {size_mb:.2f}") # muestra el resultado
    return {"size_mb": size_mb} # devuelve un diccionario con el resultado.
    # luego run_benchmark() unirá este resultado con los demás

PerformanceBenchmark.compute_size = compute_size # sustituye el método vacío de la clase por esta implementación.

In [46]:
# Queremos medir la latencia del modelo
from time import perf_counter # reloj de alta precisión. Ideal para medir un trozo de código

for _ in range(3): # repetir tres veces
    start_time = perf_counter()
    _ = pipe(query)
    latency = perf_counter() - start_time
    print(f"Latency (ms) - {1000 * latency:.3f}") # imprimo latencia y la paso a ms(milisengundos)

Latency (ms) - 27.641
Latency (ms) - 22.908
Latency (ms) - 21.184


In [47]:
import numpy as np

def time_pipeline(self, query="What is the pin number for my account?"):
    """This overrides the PerformanceBenchmark.time_pipeline() method"""
    latencies = [] # crea una lista vacía
    # Warmup
    for _ in range(10):
        _ = self.pipeline(query)
    # sólo hace que el modelo entre en temperatura porque la primera inferencia suele ser + lenta:
    #porque: reserva memoria, inicializa CUDA, carga Kernels, prepara buffers, optimizaciones internas...
    # Timed run
    for _ in range(100): # Empieza la medición
        start_time = perf_counter() # guarda el instante inicial
        _ = self.pipeline(query) # ejecuta el modelo -> inferencia completa
        latency = perf_counter() - start_time # mide la latencia dif de ahora con antes
        latencies.append(latency) # añado al listado latencies
    # Compute run statistics
    # *1000 para pasar a ms (milisegundos)
    time_avg_ms = 1000 * np.mean(latencies) # media
    time_std_ms = 1000 * np.std(latencies) # desv típica -> Mide si el modelo es estables -> indica los ms  sobre los que oscila alrededor de la media
    print(f"Average latency (ms) - {time_avg_ms:.2f} +\- {time_std_ms:.2f}")
    return {"time_avg_ms": time_avg_ms, "time_std_ms": time_std_ms}

PerformanceBenchmark.time_pipeline = time_pipeline

In [60]:
pb = PerformanceBenchmark(pipe, clinc["test"])
perf_metrics = pb.run_benchmark() # ejecuta el benchmark

AttributeError: 'InferenceSession' object has no attribute 'state_dict'

## Making Models Smaller via Knowledge Distillation

### Knowledge Distillation for Fine-Tuning

<img alt="Soft Probabilities" caption="Comparison of a hard label that is one-hot encoded (left), softmax probabilities (middle), and softened class probabilities (right)" src="images/chapter08_soft-probs.png" id="soft-probs"/> 

<img alt="Knowledge distillation" caption="The knowledge distillation process" src="images/chapter08_kd.png" id="kd"/> 

### Knowledge Distillation for Pretraining

### Creating a Knowledge Distillation Trainer

In [ ]:
# versión ampliada de TrainingArguments para guardar dos hiperparámetros nuevos:
    # alpha
    # temperature

from transformers import TrainingArguments

class DistillationTrainingArguments(TrainingArguments):
    # la nueva clase hereda todo lo que ya ofrece TrainingArguments y añade alpha y temperature
    def __init__(self, *args, alpha=0.5, temperature=2.0, **kwargs): # constructor
        # *args -> argumentos posicionales
        # *kwargs -> argumentos con nombre pertenecientes a TraingArguments (learning_rate, num_train_epochs...)
        super().__init__(*args, **kwargs) # inicializa clase padre
        # Esto ejecuta el constructor original de TrainingArguments.
        self.alpha = alpha # guarda apha, que no estaban en TrainingArguments
        self.temperature = temperature # guarda temperature, que no estaban en TraingArguments

In [ ]:
# Ahora creamos un Trainer que sabe entrenar un estudiante imitando a un profesor.
# es una implementación de knowledge distillation durante el fine-tuning.

import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer

class DistillationTrainer(Trainer): # nueva clase que hereda de Trainer
    # nueva clase reutiliza todo lo que hace HF
        # entrenamiento, evaluación, opitimizer, scheduler, checkpoints, logging...
    # sólo queremos cambiar la pérdida
    def __init__(self, *args, teacher_model=None, **kwargs): # constructor
        # teacher modelo porque el Trainer original sólo conoce un modelo.
        # teacher_model= None para que no sea un argumento obligatorio y no de error
        # Aquí necesitamos 2:
            # Teacher (congelado)
            # Student (el que aprende)
        super().__init__(*args, **kwargs) #inicializa el Trainer normal
        self.teacher_model = teacher_model # guarda el profesor

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # el Trainer normal calcula CrossEntropy.
        # Ahora sobreescribiremos a alphaLCE +(1-alpha)LKD
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # elige cuda si hay
        inputs = inputs.to(device) # pasa el batch de datos a cuda
        outputs_stu = model(**inputs) # forward del estudiante: obtiene logits y loss
        
        # Extract cross-entropy loss and logits from student
        loss_ce = outputs_stu.loss # extrae la pérdida LCE
        logits_stu = outputs_stu.logits # extrae los logits
       
        # ejecuta el Teacher
        # Extract logits from teacher
        with torch.no_grad(): # impide calcular gradientes y crear el grafo para el teacher
            # sólo pregunta -> Teacher -> Respuesta
            outputs_tea = self.teacher_model(**inputs) # outputs del teacher
            logits_tea = outputs_tea.logits # extraemos del ouutput los logits del teacher
        # Soften probabilities and compute distillation loss
        loss_fct = nn.KLDivLoss(reduction="batchmean") # Crea pérdida KLD fct= function
        # esta f de pérdida compara student vs teacher
        # estructuctura:
            #loss_kd = loss_fct(student_probs,teacher_probs)
            #para calcular prob -> softmax
            #divide por temperature para suavizar distribución -> esto reduce gradientes en 1/T**2
            # Por eso multiplica al principio por T**2
        # reduction="batchmean": suma todas las pérdidas de los ejemplos y clases y divide por el núm de ejemplos del batch
        loss_kd = self.args.temperature ** 2 * loss_fct(
            F.log_softmax(logits_stu / self.args.temperature, dim=-1),# ojo para el lado de student -> log_softmax
            F.softmax(logits_tea / self.args.temperature, dim=-1))
            # KLDivLoss(input, target)
                # input -> debe estar en lo_probabilidades
                # target -> debe estar en probabilidades normales
            #dim=-1 ->logits shape (nº ejemplos, nº de clases) ->Aplicamos softmax sobre última dim: clases 
            # las probabilidades de cada ejemplo suman 1
        # Return weighted student loss/ Combinamos pérdidas
        # LStudent = alpha*LCE + (1-alpha)*LKD
        loss = self.args.alpha * loss_ce + (1. - self.args.alpha) * loss_kd
        return (loss, outputs_stu) if return_outputs else loss
        # if return_outputs else loss ->durante entrenamiento: return_outputs=False
        # El trainer utilizará esa pérdida para hacer backward. Teacher nunca cambia

```text
                 Batch
                   │
         ┌─────────┴─────────┐
         │                   │
         ▼                   ▼
     Student             Teacher
         │                   │
         ▼                   ▼
   logits_stu          logits_tea
         │                   │
         │                   │
         ▼                   ▼
 CrossEntropy          Soft Targets
         │                   │
         └─────────┬─────────┘
                   ▼
             KL Divergence
                   │
                   ▼
      L = α·LCE + (1−α)·LKD
                   │
                   ▼
     Actualizar SOLO el Student
```

### Choosing a Good Student Initialization

In [49]:
#hide_output

# Importamos la clase que carga automáticamente el tokenizer adecuado 
# para cualquier checkpoint de Hugging Face.
from transformers import AutoTokenizer 

student_ckpt = "distilbert-base-uncased" # guardamos el nombre del modelo del estudiante
student_tokenizer = AutoTokenizer.from_pretrained(student_ckpt) # cargamos tokenizador asociado a DistilBert 
def tokenize_text(batch): # esta función recibirá un batch
    return student_tokenizer(batch["text"], truncation=True)
    # El tokenizador toma la clave "text" del batch
    # Truncation=True -> Si una frase supera la longitud máxima admitida por el modelo, se corta automáticamente.
    #evita errores coh textos largos
clinc_enc = clinc.map(tokenize_text, batched=True, remove_columns=["text"])
# crea un nuevo dataset a partir de tokenizar el dataset original
# Una vez tokenizado elimina la columna "text"
# Pasamos de tener las columnas "text" e "intent" a "input_ids", "attention_mask" e "intent"
clinc_enc = clinc_enc.rename_column("intent", "labels")
# Como Trainer espera que la variable objetivo se llame "labels",
# Cambiamos el nombre la columna "intent" por "labels"
# Después las columnas del dataset clinc_enc quedan así
    # input_ids → [101, 2338, 1037, 3462, 102]
    # attention_mask → [1, 1, 1, 1, 1]
    # labels → 42


In [ ]:
# Para saber si estoy autenticado en HF

from huggingface_hub import whoami

whoami()


In [ ]:
# from huggingface_hub import notebook_login

# notebook_login()

In [ ]:
def compute_metrics(pred):
    predictions, labels = pred # pred es una tupla que recibe el trainer
    # predictions son logits -> shape (n_ejemplos, n_clases).
    # Son logits porque Trainer sólo devuelve la salida"cruda" del modelo -> los logits
    
    predictions = np.argmax(predictions, axis=1)
    # convertimos los logits en etiquetas -> argmax -> índice del mayor logit de cada fila
    
    return accuracy_score.compute(predictions=predictions, references=labels)
    # accuracy_score.compute() pertenece a Evaluate de HF
    # Calcula la precisión entre predicciones del modelo y etiquetas reales
    # accuracy_score.compute() porque antes-> accuracy_score = evaluate.load("accuracy")

In [ ]:
# Este bloque crea el objeto con todos los hiperparámetros 
# de entrenamiento que utilizará el DistillationTrainer.


batch_size = 48 # tamaño del lote

finetuned_ckpt = "distilbert-base-uncased-finetuned-clinc"
# nombre que tendrá el modelo una vez entrenadp
student_training_args = DistillationTrainingArguments(
    output_dir=finetuned_ckpt, eval_strategy = "epoch", 
    num_train_epochs=5, learning_rate=2e-5, 
    per_device_train_batch_size=batch_size, 
    per_device_eval_batch_size=batch_size, alpha=1, weight_decay=0.01, 
    push_to_hub=True)

# student_training_args = DistillationTrainingArguments()
# crea un instancia de la clase que definimos antes -> DistillationTrainingArguments()
# donde se añadía alpha y temperature además de losTrainingArguments clásicos
# output_dir=> se guardarn en"distilbert-base-uncased-finetuned-clinc"
    # chackpoints
    # pesos
    # tokenizer
    # configuración
# evaluation_strategy="epoch" -> Evalúa el modelo al terminar cada época
# num_train_epochs=5 -> Todo el conjunto de entrenamiento se recorrerá 5 veces
# alpha=1  L=αLCE+(1−α)LKD -> No hay destilación sólo crossentropy
# El profesor no influye nada, por ahora...



In [ ]:
#hide
# on ajustes que se añadieron para que la ejecución sea más cómoda 
# o para evitar ciertos comportamientos por defecto del Trainer
# objeto.atributo = nuevo_valor

student_training_args.logging_steps = len(clinc_enc['train']) // batch_size 
# muestra el log cada "(clinc_enc['train']) // batch_size" pasos
student_training_args.disable_tqdm = False
# bRR de progreso visible
student_training_args.save_steps = 1e9
# No guardará pesos hasta que supere 1000.000.000 de pasos -> Hasta que termine el entrenamiento
student_training_args.log_level = "error"
# controla el nivel de mensajes de logging
    # DEBUG
    # INFO
    # WARNING
    # ERROR -> Sólo muestra mensajes de error importante
    # CRITICAL

In [ ]:
#hide
%env TOKENIZERS_PARALLELISM=false

# Con esto se le dice al tokenizador
# No intentes paralelizar tú. Ya se encargará PyTorch o el DataLoader.

In [ ]:
# recuperan la correspondencia entre identificadores numéricos 
# y nombres de las clases del modelo del profesor.

id2label = pipe.model.config.id2label

# recupera un diccionario como:
#{
#    0: "restaurant_reviews",
#    1: "nutrition_info",
#    2: "play_music",
#    ...
#}


label2id = pipe.model.config.label2id

# hace lo contrario

#{
#    "restaurant_reviews": 0,
#    "nutrition_info": 1,
#    "play_music": 2,
#    ...
#}

Dentro de config hay información como:

- número de etiquetas (num_labels)
- nombre del modelo
- arquitectura
- id2label
- label2id

No son pesos del modelo, sino metadatos.

In [ ]:
# Este bloque prepara la configuración del modelo estudiante, 
# pero todavía no crea el modelo. Esa diferencia es muy importante.

from transformers import AutoConfig

num_labels = intents.num_classes # número de clases del dataset CLINC.
student_config = (AutoConfig
                  .from_pretrained(student_ckpt, num_labels=num_labels, 
                                   id2label=id2label, label2id=label2id))

# student_config NO es el modelo sino una colección de parámetros

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification # crea un modelo de clasificación a partir de un checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# se define una función para crear el modelo estudiante

def student_init():
    return (AutoModelForSequenceClassification
            .from_pretrained(student_ckpt, config=student_config).to(device))

In [ ]:
#hide_output
teacher_ckpt = "transformersbook/bert-base-uncased-finetuned-clinc"
teacher_model = (AutoModelForSequenceClassification
                 .from_pretrained(teacher_ckpt, num_labels=num_labels)
                 .to(device))

¿Por qué aquí no existe teacher_init() y hacemos teacher_model=...?

Razón -> papel de cada uno:

- Estudiante: Va a cambiar continuamente. Epoch1-> Pesos cambian -> Epoch2 -> pesos cambian

  Por eso el Trainer necesita crear un estidante nuevo cuando haga falta

- Profesor -> Nunca cambia


                 Teacher
      bert-base-finetuned-clinc
                 │
                 ▼
           teacher_model
                 │
         (NO cambia nunca)
                 │
                 ▼
         Produce logits_teacher
                 │
                 │
                 ▼
Student  ───────────────► KL Divergence
                 ▲
                 │
          logits_student
                 │
          Pesos se actualizan

In [ ]:
# Aquí se juntan todas piezas

# se crea el DistillationTrainer
distilbert_trainer = DistillationTrainer(model_init=student_init,
    teacher_model=teacher_model, args=student_training_args,
    train_dataset=clinc_enc['train'], eval_dataset=clinc_enc['validation'],
    compute_metrics=compute_metrics, tokenizer=student_tokenizer)
# No se pasa un modelo estudiante ya creado sino la función student_init SIN PARÉNTESIS

# ejecuta el entrenamiento
distilbert_trainer.train()

In [ ]:
student_training_args.output_dir


In [ ]:
#hide_output
distilbert_trainer.push_to_hub("Training completed!")


In [ ]:
#hide_output

# Ahora que lo tenemos guardado en The Hub podemos usarlo en un pipeline

finetuned_ckpt = "transformersbook/distilbert-base-uncased-finetuned-clinc"
pipe = pipeline("text-classification", model=finetuned_ckpt)

In [ ]:
type(pipe.model)

In [ ]:
pipe.model.config.model_type

In [ ]:
# Hacemos benchmark de Distilbert
optim_type = "DistilBERT"

pb = PerformanceBenchmark(
    pipe,
    clinc["test"],
    optim_type=optim_type
)

perf_metrics.update(pb.run_benchmark())

In [ ]:
import pandas as pd # Importamos pandas para crear un DF porque es
# Más cómodo que recorrer un diccionario

def plot_metrics(perf_metrics, current_optim_type):
    df = pd.DataFrame.from_dict(perf_metrics, orient='index')
    # crea un df a partir del diccionario perf_metrics
    # orient='index' -> Las claves del diccionario serán el índice

    for idx in df.index: # recorre todos los modelos
        df_opt = df.loc[idx] # obtiene una fila (la del índice que iteramos)
        # Add a dashed circle around the current optimization type
        if idx == current_optim_type:
            plt.scatter(df_opt["time_avg_ms"], df_opt["accuracy"] * 100, 
                        alpha=0.5, s=df_opt["size_mb"], label=idx, 
                        marker='$\u25CC$')
            # plt.scatter() dibuja un punto en un plano
            # s=df_opt["size_mb"] -> el tamaño dela burbuja depende del tamaño del modelo
            # alpha=0.5 -> Burbuja semi transparente
            # label=idx -> texto de la leyenda
            # marker='$\u25CC$'-> caracter unicode -> Círculo punteado
        else: # si no es el current_optim_type
            plt.scatter(df_opt["time_avg_ms"], df_opt["accuracy"] * 100, 
                        s=df_opt["size_mb"], label=idx, alpha=0.5)
            
    legend = plt.legend(bbox_to_anchor=(1,1)) # genera caja para la leyenda
    for handle in (legend.legend_handles if hasattr(legend, "legend_handles") else legend.legendHandles):
        handle.set_sizes([20]) # el tamaño de los puntos en la leyenda será 20 independientemente del tamaño que tengan en el gráfico

    plt.ylim(80,90)
    # Use the slowest model to define the x-axis range
    
    xlim = int(perf_metrics["BERT baseline"]["time_avg_ms"] + 3)
    plt.xlim(1, xlim)
    plt.ylabel("Accuracy (%)")
    plt.xlabel("Average latency (ms)")
    plt.show()
    
plot_metrics(perf_metrics, optim_type)

### Compromiso entre rendimiento, velocidad y tamaño

En la gráfica se aprecia claramente el compromiso (*trade-off*) que ofrece **DistilBERT** frente a **BERT**.

#### 🔵 BERT (baseline)

- Mayor **accuracy** (~86,7 %).
- Más lento (~7,3 ms por inferencia).
- Modelo más pesado (~418 MB).

#### 🟠 DistilBERT

- Pierde muy poca **accuracy** (~85,8 %).
- Es considerablemente más rápido (~4 ms por inferencia).
- Ocupa bastante menos (~256 MB).

---

### Comparación

| Modelo | Accuracy | Tiempo de inferencia | Tamaño |
|---------|---------:|---------------------:|--------:|
| **BERT** | ~86,7 % | ~7,3 ms | ~418 MB |
| **DistilBERT** | ~85,8 % | ~4 ms | ~256 MB |

---

### Conclusión

DistilBERT consigue un excelente equilibrio entre precisión y eficiencia:

- 📉 **Pierde menos de un 1 % de accuracy.**
- ⚡ **Reduce el tiempo de inferencia casi a la mitad.**
- 💾 **Disminuye significativamente el tamaño del modelo.**

En muchas aplicaciones reales, esta pequeña pérdida de precisión compensa ampliamente la mejora en velocidad y consumo de memoria.

### Finding Good Hyperparameters with Optuna

In [ ]:
#hide_input
#id banana-function
#alt A banana plot
#caption Plot of the Rosenbrock function of two variables 
import matplotlib.pyplot as plt
import numpy as np

def f(x, y):
    return (1-x)**2+100*(y-x**2)**2
    
X, Y = np.meshgrid(np.linspace(-2, 2, 250), np.linspace(-1, 3, 250))
Z = f(X,Y)
_, ax = plt.subplots()
ax.plot([1], [1], 'x', mew=3, markersize=10, color="red")
ax.contourf(X, Y, Z, np.logspace(-1, 3, 30), cmap='viridis', extend="both")
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-0.9, 1.7)
plt.show()

In [ ]:
# idea -> Si pruebas estos valores, qué resultados obtienes?

def objective(trial):
    x = trial.suggest_float("x", -2, 2) # optuna elige para x un valor entre -2  y 2
    y = trial.suggest_float("y", -2, 2) # optuna elige para y un valor entre -2  y 2
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2 # se evalúa la función con los números elegidos

   # trial.suggest_float(): Pide al algoritmo de Optuna un valor para esa variable 

optuna recopila multiples trials como un study. 

Para crear un study, tenemos que pasar la función objective() a study.optimize() 

In [ ]:
#hide_output
# 
import optuna # carga la librería Optuna

study = optuna.create_study()
# se crea un objeto llamado Study. 
# Un Study representa un experimento completo de optimización
# Optuna guarda aquí:
    # todos los intentos (trials)
    # parámetros usados
    # valor obtenido
    # cuál ha sido el mejor hasta ahora
# study gurda todo el proceso de búsqueda


study.optimize(objective, n_trials=1000)

# aquí empieza la búsqueda
# ejecuta la función objective() 1000 veces

In [ ]:
study.best_params # mejores parámetros

In [ ]:
# Optuna llamará a esta función una vez por cada trial.

def hp_space(trial):
    return {"num_train_epochs": trial.suggest_int("num_train_epochs", 5, 10),
        "alpha": trial.suggest_float("alpha", 0, 1),
        "temperature": trial.suggest_int("temperature", 2, 20)}
# La función devuelve un diccionario.
# Ese diccionario contiene los hiperparámetros que se usarán 
# para entrenar el modelo en ese trial.
# Por ejemplo:
    
#{
#    "num_train_epochs": 7,
#    "alpha": 0.42,
#    "temperature": 11
#}

In [ ]:
#hide_output
best_run = distilbert_trainer.hyperparameter_search(
    n_trials=20, direction="maximize", hp_space=hp_space)

# Entrena el modelo 20 veces con distintos hiperparámetros 
# y quédate con la combinación que obtenga el mejor resultado."

# aquí el Trainer se creó con compute_metrics= compute_metrics -> que devuelve "accuracy"
# quéremos que la accuracy sea "máxima"

In [ ]:
print(best_run)

In [ ]:
#hide_output
# usamos los mejores hiperparámetros encontrados por Optuna 
# para entrenar un nuevo modelo destilado definitivo.
# actualizamos los traing_args

for k,v in best_run.hyperparameters.items(): # recorre las claves y valores del diccionario
    setattr(student_training_args, k, v)
    # setattr() -> asigna dinámicamente esos valores al objeto student_training_args
    
# Define a new repository to store our distilled model /carpeta del modelo definitivo
distilled_ckpt = "distilbert-base-uncased-distilled-clinc"
student_training_args.output_dir = distilled_ckpt

# Create a new Trainer with optimal parameters
distil_trainer = DistillationTrainer(model_init=student_init,
    teacher_model=teacher_model, args=student_training_args,
    train_dataset=clinc_enc['train'], eval_dataset=clinc_enc['validation'],
    compute_metrics=compute_metrics, tokenizer=student_tokenizer)

distil_trainer.train();

In [ ]:
#hide_output
distil_trainer.push_to_hub("Training complete")

### Benchmarking Our Distilled Model

In [ ]:
distilled_ckpt = "transformersbook/distilbert-base-uncased-distilled-clinc"
pipe = pipeline("text-classification", model=distilled_ckpt)
optim_type = "Distillation"
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())

In [ ]:
plot_metrics(perf_metrics, optim_type)

## Making Models Faster with Quantization

### Sidebar: A Primer on Floating-Point and Fixed-Point Numbers

### End sidebar

<img alt="Mapping floating-point numbers to 8-bit integers" width="800" caption="Quantizing floating-point numbers as unsigned 8-bit integers (courtesy of Manas Sahni)" src="images/chapter08_fp32-to-int8.png" id="fp32toint8"/>

In [ ]:
# inspección de los pesos del modelo antes de cuantizarlo

import matplotlib.pyplot as plt

state_dict = pipe.model.state_dict() # obtiene todos los pesos del modelo
weights = state_dict["distilbert.transformer.layer.0.attention.out_lin.weight"]
# elegimos una matriz de pesos concreta -> layer 0
plt.hist(weights.flatten().cpu().numpy(), bins=250, range=(-0.3,0.3), edgecolor="C0")
# dibuja histogrna vidido en 250 cajas en un rango de -0.3 a 0.3
# weights.flatten()-> convierte la matriz en un único vector -> nºs uno detrás de otrs
# .cpu() -> Porque el tensor está en la GPU y matplotlib sólo lee en CPU. Se pasa a CPU
# . numpy() -> convierte en un array de Numpy
plt.show()

In [ ]:
# Calculamos los dos parámetros que permiten pasar de números en coma flotante (float32) 
# a enteros de 8 bits (int8).

zero_point = 0
# El zero point indica qué valor entero representa al 0 real.
# En este caso el 0 real coincide con el 0 entero. Cuantización simétrica

scale = (weights.max() - weights.min()) / (127 - (-128))
# calcula el factor de escala -> Cuánto vale un paso de un entero int8 en el mundo real?

In [ ]:
# Realizamos la cuantización propiamente dicha
# f= q/z + s
# .clamp() limita el tensor entre un mínimo y un maximo -> límites del integer
# tensor.clamp(min, max) -> si un valor es menor que min o mayor que max los sustituye
# por min y max
# .char() -> necesario para convertir el tensor al tipo torch.int8
(weights / scale + zero_point).clamp(-128, 127).round().char()

Ojo! Sólo hemos cuantizado la matriz de pesos de una capa.

Esto es **sólo una demostración**

In [ ]:
# Hace exactamente lo mismo que hice manualmente antes, 
# pero utilizando la API oficial de cuantización de PyTorch.

from torch import quantize_per_tensor # función que cuantiza un tensor completo.

dtype = torch.qint8 # convertimos en un "tensor cuantizado" -> almacena int8 + scale + zero_point
quantized_weights = quantize_per_tensor(weights, scale, zero_point, dtype) # cuantizamos
quantized_weights.int_repr() # para ver los enteros almacenados

In [ ]:
#hide_input
#id weight-quantization
#alt Effect of quantizati
#on a transformer's weights
#caption Effect of quantization on a transformer's weights
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes,mark_inset

# Create histogram
fig, ax = plt.subplots()
ax.hist(quantized_weights.dequantize().cpu().flatten().numpy(), 
         bins=250, range=(-0.3,0.3), edgecolor="C0");
# Create zoom inset
axins = zoomed_inset_axes(ax, 5, loc='upper right')
axins.hist(quantized_weights.dequantize().cpu().flatten().numpy(), 
         bins=250, range=(-0.3,0.3));
x1, x2, y1, y2 = 0.05, 0.1, 500, 2500
axins.set_xlim(x1, x2)
axins.set_ylim(y1, y2)
axins.axes.xaxis.set_visible(False)
axins.axes.yaxis.set_visible(False)
mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.5")
plt.show()

Cuando cuantizamos y descuantizamos perdemos información por eso se generan algunos huecos en el gráfico


Si cuantizo estos Valores float

- 0.03111
- 0.03118
- 0.03126
- 0.03134
- 0.03143
- 0.03152
  
y los vuelvo a descuantizar podrían quedar así:

- 0.0314
- 0.0314
- 0.0314
- 0.0314
- 0.0314
- 0.0314

Ya no hay continuidad. Sólo hay niveles discretos


**Idea**
- La forma general de la distribución se conserva.
- Pero, al ampliar la imagen, se ve que ya no es continua, sino que está formada por escalones discretos. Ese es el verdadero efecto visual de la cuantización.

In [ ]:
%%timeit 

# multiplicamos tensores FP 32
weights @ weights

# @ -> multiplicación matricial -> equivale a torch.matmul(weights, weights)

In [ ]:
# Para multiplicar tensores cuantozados (INT8)
# @ y matmul no están definidos para tensores cuantizados -> QFunctional

from torch.nn.quantized import QFunctional

# clase especializada para realizar operaciones matemáticas sobre tensores cuantizados.

q_fn = QFunctional() # creamos objeto

In [ ]:
# Llevo primero los pesos a la CPU

weights_cpu = weights.detach().cpu()

zero_point = 0
scale = float((weights_cpu.max() - weights_cpu.min()) / (127 - (-128)))

quantized_weights_cpu = torch.quantize_per_tensor(
    weights_cpu,
    scale=scale,
    zero_point=zero_point,
    dtype=torch.qint8
)

In [ ]:
%%timeit

q_fn.mul(quantized_weights_cpu, quantized_weights_cpu)
# Multiplicamos posición por posición -> aij x aij

In [ ]:
import sys

sys.getsizeof(weights_cpu.storage()) / sys.getsizeof(quantized_weights_cpu.storage())

Como cada valor ocupa aproximadamente:

float32 → 4 bytes

int8    → 1 byte

**Los pesos cuantizados ocupan aproximadamente 4 veces menos memoria**

## Diferencia importante entre los distinos tipos de cuantización
La diferencia importante

**No es tanto cómo cuantizan, sino cuándo toman las decisiones**

- Dynamic -> Voy decidiendo durante la inferencia.

- Static -> Lo dejo todo preparado antes de la inferencia.

- QAT -> Enseño al modelo desde el entrenamiento que vivirá en INT8.

In [ ]:
#hide_output

# Implementación práctica de la cuantización dinámica

# Importamos la función que automatiza la cuantización dinámica
# hasta ahora manualmente: weights -> calcular scale -> calcular zero_point -> quantize-per_tensor()
# quantize_dynamic -> sustituye todo esto

from torch.quantization import quantize_dynamic

# Guardamos el modelo que vamos a descargar
model_ckpt = "transformersbook/distilbert-base-uncased-distilled-clinc"

# cargamos tokenizador en base al modelo
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
# cargamos el modelo
model = (AutoModelForSequenceClassification
         .from_pretrained(model_ckpt).to("cpu")) # hay que mover el modelo al CPU para cuantizar

#Aplicamos cuantización dinámica
model_quantized = quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
    # {nn.Linear} -> cuantiza sólo las capas de tipo nn.linear (layernorm, Embedding, Droput... las deja igual.
    # dtype=torch.qint8 -> usa entero de 

# ¿Qué hace internamente quantize_dynamic?
```text
Modelo FP32
        │
        ▼
Recorrer todas las capas
        │
        ├──────── Embedding
        │            │
        │            ▼
        │        No tocar
        │
        ├──────── LayerNorm
        │            │
        │            ▼
        │        No tocar
        │
        ├──────── Linear
        │            │
        │            ▼
        │     Cuantizar a INT8
        │
        ├──────── Linear
        │            │
        │            ▼
        │     Cuantizar a INT8
        │
        ▼
Nuevo modelo
``` 

### Benchmarking Our Quantized Model

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from torch.quantization import quantize_dynamic
model_ckpt = "transformersbook/distilbert-base-uncased-distilled-clinc"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

## comprobamos si la cuantización ha merecido la pena.

# Cargar una copia nueva directamente en CPU
model_cpu = AutoModelForSequenceClassification.from_pretrained(model_ckpt)
model_cpu = model_cpu.cpu()
model_cpu.eval()

# Cuantizar la copia que está en CPU
model_quantized = quantize_dynamic(
    model_cpu,
    {nn.Linear},
    dtype=torch.qint8
)




pipe = pipeline("text-classification", model=model_quantized, 
                tokenizer=tokenizer, device=-1)
# device =-1 Obligamos a que el pipeline vaya por CPU.  =0->GPU
optim_type = "Distillation + quantization"
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())

In [ ]:
plot_metrics(perf_metrics, optim_type)

In [ ]:
# Para restaurar print
import builtins

import builtins
print = builtins.print

In [ ]:
import torch
print(torch.backends.quantized.engine)

In [ ]:
import transformers
print(transformers.__version__)
print(torch.__version__)

## Optimizing Inference with ONNX and the ONNX Runtime

<img alt="Example ONNX graph" width="500" caption="A section of the ONNX graph for BERT-base, visualized in Netron" src="images/chapter08_bert-onnx.png" id="bert-onnx"/>

<img alt="Architecture of the ONNX and ONNX Runtime ecosystem" width="500" caption="Architecture of the ONNX and ONNX Runtime ecosystem (courtesy of the ONNX Runtime team)" src="images/chapter08_onnx-ort.png" id="onnx-ort"/>

In [ ]:
#hide_output

# Preparamos el entorno para que ONNX Runtime (ORT) aproveche bien la CPU

import os
# módulo que permite trabajar con el sistema operativo -> modificar variables de entorno
    # variables de entorno -> "configuración global" que los programas leen cuando arrancan.

from psutil import cpu_count
# psutil puede consultar información del sistema.
# cpu_count -> número de hilos lógicos

os.environ["OMP_NUM_THREADS"] = f"{cpu_count()}"
# Le decimos a OpenMP -> "Puedes utilizar todos los hilos disponibles."
# OpenMP -> biblioteca para paralelizar cálculos

os.environ["OMP_WAIT_POLICY"] = "ACTIVE"
# mantiene los hilos activos


In [ ]:
cpu_count()

- 8 P-cores (Performance Cores) con Hyper-Threading → 16 hilos
- 8 E-cores (Efficient Cores) → 8 hilos

In [6]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

In [7]:
from pathlib import Path

In [8]:
# Para no tener que ejecutar celdas anteriores
from transformers import AutoTokenizer

model_ckpt = "transformersbook/distilbert-base-uncased-distilled-clinc"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [10]:
#hide_output
# Creamos el archivo ONNX
import torch
from transformers.convert_graph_to_onnx import convert
# Importamos la función "convert" q sabe traducir un modelo de Transformers al formato ONNX.

torch.set_default_device("cpu")
#forzamos a trabajar con la cpu no con la GPU

model_ckpt = "transformersbook/distilbert-base-uncased-distilled-clinc"
onnx_model_path = Path("onnx_models/model.onnx") 
# definimos el archivo de salida
onnx_model_path.parent.mkdir(parents=True, exist_ok=True)
# si no existe esa carpeta, la crea
# Este .onnx contiene:
    # el grafo computacional
    # los pesos del modelo




In [11]:
from pathlib import Path
import shutil

local_onnx_dir = Path("onnx")

if local_onnx_dir.exists():
    shutil.rmtree(local_onnx_dir)

In [12]:
convert(framework="pt", model=model_ckpt, tokenizer=tokenizer, 
        output=onnx_model_path, opset=14, pipeline_name="text-classification")
# opset=12 -> Un opset es una versión del conjunto de operadores de ONNX.

# Ojo! Durante esta conversión no cambia nada del modelo -> sólo la forma de representarlo
# Antes -> modelo PyTorch
# Ahora -> modelo ONNX

/home/srmjf/qa/lib/python3.9/site-packages/transformers/convert_graph_to_onnx.py:361: FutureWarning: The `transformers.convert_graph_to_onnx` package is deprecated and will be removed in version 5 of Transformers
  warnings.warn(


ONNX opset version set to: 14
Loading pipeline (model: transformersbook/distilbert-base-uncased-distilled-clinc, tokenizer: DistilBertTokenizerFast(name_or_path='transformersbook/distilbert-base-uncased-distilled-clinc', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=F

Device set to use cpu


Using framework PyTorch: 2.7.1+cu118
Found input input_ids with shape: {0: 'batch', 1: 'sequence'}
Found input attention_mask with shape: {0: 'batch', 1: 'sequence'}
Found output output_0 with shape: {0: 'batch'}
Ensuring inputs are in correct order
head_mask is not present in the generated input list.
Generated inputs order: ['input_ids', 'attention_mask']


In [15]:
# Esta función crea una sesión de inferencia de ONNX Runtime a partir del archivo .onnx. 
# Pasa d tener el modelo guardado a tenerlo preparado xa recibir entradas y dar predicciones.

from onnxruntime import (GraphOptimizationLevel, InferenceSession, 
                         SessionOptions)
# InferenceSession: carga y ejecuta el modelo ONNX.
# SessionOptions: configura cómo se ejecutará.
# GraphOptimizationLevel: indica cuánto debe optimizar ONNX Runtime el grafo.


def create_model_for_provider(model_path, provider="CPUExecutionProvider"): 
    
    options = SessionOptions()
    # Crea un objeto donde se almacenan las decisiones de configuración de la sesión.
    options.intra_op_num_threads = 1
    # Aquí se establece el paralelismo dentro de una operación.
    # Establece que si hay una operacion grnde -> sólo la hará un hilo
    options.graph_optimization_level = GraphOptimizationLevel.ORT_ENABLE_ALL
    # Indica a ONNX Runtime que aplique todas las optimizaciones disponibles.
    
    
    session = InferenceSession(str(model_path), options, providers=[provider])
    # crea la sesión
    # provider -> lista porque puedes poner varios providers
        #providers=["CUDAExecutionProvider","CPUExecutionProvider"]
    session.disable_fallback()
    # Desactiva ese cambio automático a otro provider.
    # útil para un benchmark porque garantiza que el modelo está ejecutándose realmente 
    # con el provider que querías medir. Evita creer que estás midiendo GPU cuando parte del trabajo se ha desplazado silenciosamente a CPU.
    return session
    # La función devuelve el objeto preparado para inferencia.

In [16]:
# Dejamos de trabajar con un archivo y pasas a trabajar con un modelo ejecutable.

onnx_model = create_model_for_provider(onnx_model_path)

# aquí onnx_model no es un modelo de PyTorch. No es un nn.Module
#Es una: onnxruntime.InferenceSession

In [17]:
type(onnx_model)

onnxruntime.capi.onnxruntime_inference_collection.InferenceSession

- Hasta ahora: estábamos preparando el modelo ONNX.
- A partir de aquí: empezamos a utilizar ONNX Runtime para hacer inferencia y, más adelante, comparar su rendimiento con PyTorch.

In [51]:
inputs = clinc_enc["test"][:1] # primer ejemplo de test

# inputs contiene algo así:
#{
#    "input_ids": ...,
#    "attention_mask": ...,
#    "labels": ...
#}


del inputs["labels"] #quitamos la col etiqueta porque ONNX va a predecir no a entrenar
logits_onnx = onnx_model.run(None, inputs)[0] # ejecuta el modelo
# primer argumento -> qué salidas quieres devolver? None -> Todas
# segundo argumento -> entradas "input_ids" y "attention_mask"
# run devuelve una lista de salidas
# [
#    logits,
#    hidden_states,
#    attentions
#]

# con [0] eligen la primera salida-> Los logits
logits_onnx.shape
# shape = (1,51) 1 ejemplo 150 clases


(1, 151)

In [53]:
# si queremos obtener la label predicha
np.argmax(logits_onnx)

61

In [54]:
clinc_enc["test"][0]["labels"]

61

ONNX no es compatible con la pipeline de text-classification

Hay que crear una nueva clase que la imite



## Creación de un pipeline para ONNX Runtime

Hasta ahora tenía dos piezas separadas:

```text
Texto
   │
   ▼
Tokenizer
   │
   ▼
Tensor
   │
   ▼
ONNX Runtime
   │
   ▼
Logits
```

La clase `OnnxPipeline` une todo ese proceso para que pueda ejecutar simplemente:

```python
onnx_pipeline("Book a flight to Paris")
```

igual que hacía con `pipeline()` de Transformers.

---


In [56]:
# Esta clase es, conceptualmente, la versión ONNX de un pipeline de Hugging Face.
from scipy.special import softmax

class OnnxPipeline:
    def __init__(self, model, tokenizer): # constructor
        self.model = model # el modelo ONNX
        self.tokenizer = tokenizer
        
    def __call__(self, query): # gracias a call la clase se comporta como una función
        model_inputs = self.tokenizer(query, return_tensors="pt")
        # tokeniza convierte entradas en input_ids y attention_mask
        inputs_onnx = {k: v.cpu().detach().numpy() 
                       for k, v in model_inputs.items()}
        # ONNX Runtime no trabaja con tensores de PyTorch sino con arrays de Numpy
        #.cpu() -> Si el tensor estuviera en GPU:lo envía a la CPU
        #.detach() -> Rompe el vínculo con el sistema de cálculo de gradientes de PyTorch.
                    # Como estamos haciendo inferencia, no necesitamos gradientes.
        # .numpy() ->Convierte finalmente el tensor en un array de NumPy
            #{"input_ids": numpy.ndarray,"attention_mask": numpy.ndarray}

        
        logits = self.model.run(None, inputs_onnx)[0][0, :]
        # ejecuta ONNX
        # De los logits [0] quiero la primera frase [0] y todas las clases [0, :]
        probs = softmax(logits)
        # convertimos a probabilidades
        pred_idx = np.argmax(probs).item()
        # posición con la probabilidad mayor
        return [{"label": intents.int2str(pred_idx), "score": probs[pred_idx]}]

In [57]:
pipe = OnnxPipeline(onnx_model, tokenizer) # objeto del pipeline
pipe(query) # lo ejcutamos

[{'label': 'car_rental', 'score': 0.78483325}]

Creamos un benchmark que hereda del la clase benchmark que hicimos antes.

Hay que sobreescribir la parte de compute_size() porque ONNX no tiene acceso a los atributos nn.module de PyTorch y por lo tanto no puede acceder a state_dict ni a torch.save() 

In [58]:
class OnnxPerformanceBenchmark(PerformanceBenchmark):
    def __init__(self, *args, model_path, **kwargs):
        super().__init__(*args, **kwargs)
        self.model_path = model_path
        
    def compute_size(self):
        size_mb = Path(self.model_path).stat().st_size / (1024 * 1024)
        print(f"Model size (MB) - {size_mb:.2f}")
        return {"size_mb": size_mb}

In [59]:
optim_type = "Distillation + ORT"
pb = OnnxPerformanceBenchmark(pipe, clinc["test"], optim_type,
                              model_path="onnx/model.onnx")
perf_metrics.update(pb.run_benchmark())

NameError: name 'perf_metrics' is not defined

In [ ]:
plot_metrics(perf_metrics, optim_type)

In [ ]:
# Para hacer cuantización dinámica en ONNX

from onnxruntime.quantization import quantize_dynamic, QuantType

model_input = "onnx/model.onnx" # modelo den entrada, el modelo ONNX original
model_output = "onnx/model.quant.onnx" # cómo se va a llamar y dónde va air el modelo de salida
quantize_dynamic(model_input, model_output, weight_type=QuantType.QInt8) # crea el modelo

In [ ]:
onnx_quantized_model = create_model_for_provider(model_output) 
# crea una session para inferencia a partir del modelo cuantificado
pipe = OnnxPipeline(onnx_quantized_model, tokenizer)
optim_type = "Distillation + ORT (quantized)"
pb = OnnxPerformanceBenchmark(pipe, clinc["test"], optim_type, 
                              model_path=model_output)
perf_metrics.update(pb.run_benchmark())

In [ ]:
plot_metrics(perf_metrics, optim_type)

## Making Models Sparser with Weight Pruning

### Sparsity in Deep Neural Networks

<img alt="Network Pruning" width="500" caption="Weights and neurons before and after pruning (courtesy of Song Han)" src="images/chapter08_network-pruning.png" id="network-pruning"/> 

### Weight Pruning Methods

#### Magnitude pruning

In [ ]:
#hide_input
#id sparsity-scheduler
#alt Sparsity scheduler
#caption The cubic sparsity scheduler used for pruning
import numpy as np
import matplotlib.pyplot as plt

def _sparsity(t, t_0=0, dt=1, s_i=0, s_f=0.9, N=100):
    return s_f + (s_i - s_f) * (1 - (t - t_0) / (N * dt))**3

steps = np.linspace(0,100,100)
values = [_sparsity(t) for t in steps]

fig, ax = plt.subplots()
ax.plot(steps, values)
ax.set_ylim(0,1)
ax.set_xlim(0,100)
ax.set_xlabel("Pruning step")
ax.set_ylabel("Sparsity")
plt.grid(linestyle="dashed")
plt.show()

#### Movement pruning

<img alt="Magnitude vs Movement Pruning" width="700" caption="Comparison of weights removed (in gray) during magnitude pruning (left) and movement pruning (right)" src="images/chapter08_magnitude-vs-movement.png" id="magnitude-vs-movement"/> 

<img alt="Pruning Distributions" width="500" caption="Distribution of remaining weights for magnitude pruning (MaP) and movement pruning (MvP)" src="images/chapter08_pruning-dists.png" id="pruning-dists"/>

## Conclusion